# 02 · GPU experiments on Kaggle

Runs the GPU-heavy steps when the free Colab GPU quota is used up. The code is the same repository; only the paths differ.

**One-time preparation**
1. Kaggle account with a verified phone number (needed for GPU and internet).
2. Upload the four processed Parquet files (`unsw`, `toniot`, `botiot`, `cic2018`) as a private Kaggle dataset and attach it with *Add Input*.
3. *Add-ons → Secrets*: add `GH_TOKEN` (read-only GitHub token) and switch it on for this notebook.
4. *Settings*: Accelerator **GPU T4 x2** (or P100), Internet **on**.

Results are written to `/kaggle/working/results` and zipped at the end; copy them into `MyDrive/thesis_data/results` so every result stays in one place.

## Step 0 · Setup

In [ ]:
import os, glob, shutil, subprocess
from kaggle_secrets import UserSecretsClient

R = '/kaggle/working/netflow-crossdataset-ids'
if not os.path.exists(R):
    url = 'https://' + UserSecretsClient().get_secret('GH_TOKEN') + '@github.com/mirmumin/netflow-crossdataset-ids.git'
    subprocess.run(['git', 'clone', '-q', url, R], check=True)   # token is never printed
os.chdir(R)
!git pull -q origin main && git log --oneline -1

# processed data from the attached Kaggle dataset
os.makedirs('data/processed', exist_ok=True)
files = glob.glob('/kaggle/input/**/*.parquet', recursive=True)
for f in files:
    shutil.copy(f, 'data/processed/')
print(sorted(os.path.basename(f) for f in files))

!pip install -q tabpfn==9.2.0 pyyaml
!nvidia-smi --query-gpu=name,memory.total --format=csv
RES = '/kaggle/working/results'
os.makedirs(RES, exist_ok=True)

## Step 1 · Benign-only target: per-flow scores, ROC curves and confusion matrices (seed 42, k = 1000)

In [ ]:
!python scripts/run_benign_only.py --results-dir {RES} --save-scores --budgets 1000
!python scripts/plot_roc_cm.py {RES}

## Step 2 · Setting B: few-shot fusion vs TabPFN vs XGBoost, all five seeds
Resumable: rerun the cell after a timeout and finished runs are skipped.

In [ ]:
!python scripts/run_fusion.py --results-dir {RES} --all-seeds

## Step 3 · Pack the results for Google Drive

In [ ]:
shutil.make_archive('/kaggle/working/kaggle_results', 'zip', RES)
print('Download /kaggle/working/kaggle_results.zip (Output tab) and unzip it into MyDrive/thesis_data/results')